## 10 · Zenodo release of the derived literature artifacts

Stages the Phase 1–3 outputs (tidy corpus, hotspot analytics, coder models) for publication to the **biofairnet** community, with `isDerivedFrom` links to the two source DOIs and **CC-BY-4.0** (required: the payload derives from CC-BY-4.0 data by Guerreschi, Lomuscio & Albanese).

**Safety flow** — nothing here uploads anything:
1. Stage (this notebook) → review the payload + analytics outputs
2. Commit + tag `zenodo-ul-*` → workflow uploads to **sandbox** (`use_sandbox: true` is the staged default; needs the `ZENODO_SANDBOX_TOKEN` repo secret)
3. Check the sandbox record, set `use_sandbox: false` in `metadata/zenodo_params.json`, commit, tag again → production DOI

<div class="alert alert-block alert-warning"><b>Trigger via tag, not the Actions UI:</b> manual workflow-dispatch runs fill the form with old defaults which override the params file (inputs take precedence). A tag push leaves inputs empty, so the staged params drive the release.</div>

In [ ]:
import sys
from pathlib import Path

# Make the repo importable: `helper` at the root, `gif` under `src/`.
repo_root = Path.cwd().parents[0]
for p in (str(repo_root), str(repo_root / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from gif.lit_release import stage_literature_release

### Stage payload + params

In [ ]:
report = stage_literature_release()   # sandbox default
print(f"Payload: {len(report['copied'])} files -> {report['payload_dir']}")
print("Missing :", report["missing"] or "none")
print("Sandbox :", report["use_sandbox"], "| license:", report["license"])
print("Derived from:", *report["related_dois"])

### Review checklist

- [ ] Analytics tables/figures in `data/results/literature/` are correct
- [ ] Payload listing below contains exactly what should be public
- [ ] `metadata/zenodo_params.json`: title, description, creators OK
- [ ] `ZENODO_SANDBOX_TOKEN` secret exists in the GitHub repo

In [ ]:
for name in sorted(report["copied"]):
    print("  -", name)

### Trigger the upload (run only after the review!)

<div class="alert alert-block alert-danger"><b>Attention:</b> this commits the payload and pushes a <code>zenodo-ul-*</code> tag, which starts the upload workflow (sandbox or production per <code>use_sandbox</code> in the params file).</div>

In [ ]:
TRIGGER = False   # set to True after completing the review checklist

if TRIGGER:
    import shlex, subprocess
    from datetime import datetime

    def run(cmd, check=True):
        print("$", cmd)
        subprocess.run(shlex.split(cmd), cwd=repo_root, check=check)

    run("git add notebooks/release_payload metadata/zenodo_params.json")
    run('git commit -m "chore: stage literature Zenodo release payload"', check=False)
    run("git push origin main", check=False)
    tag = "zenodo-ul-literature-" + datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    run(f"git tag {tag}")
    run("git push origin --tags")
    print(f"🚀 Triggered upload with tag: {tag} — watch GitHub → Actions")
else:
    print("Review pending — set TRIGGER = True when ready.")